In [ ]:
import sys
import os
import platform
import importlib
import glob

import numpy as np
import pandas as pd
import netCDF4
import datetime
import matplotlib.pyplot as plt

if platform.system() == 'Darwin':  # macOS
    base_FP = '/Users/hyunglokkim/Insync/hkim@geol.sc.edu/Google_Drive'
    cpuserver_data_FP = '/Users/hyunglokkim/cpuserver_data'
else:
    base_FP = '/data'
    cpuserver_data_FP = '/data'
sys.path.append(base_FP + '/python_modules')

#hydroAI libs
import HydroAI.Plot as Plot
import HydroAI.Data as Data
importlib.reload(Plot);
importlib.reload(Data);

# Ignore runtime warnings
import warnings
warnings.filterwarnings("ignore")

# Define your directory where to save nc files
nc_save_dir = cpuserver_data_FP + '/extracted_nc'

## 1. Load Resampled Data, Select Time-period, and convert ASCAT SWI to VSM

In [ ]:
# Define the research grid (SMAP EASE2 36-km)
mission_product = 'SPL3SMP.008'
domain_lon, domain_lat = hSMAP.get_e2grid(cpuserver_data_FP, mission_product)

## 2. Specify the time-period

In [ ]:
start_date = datetime.datetime(2022, 4, 1)
end_date   = datetime.datetime(2022, 10, 30)

# Calculate the day of year values
doy1 = start_date.timetuple().tm_yday
doy2 = end_date.timetuple().tm_yday

## 3. Load the Datasets

In [ ]:
nc_file_Porosity = nc_save_dir+'/PORO_R_SPL3SMP.008.nc'
nc_data_PORO   = netCDF4.Dataset(nc_file_Porosity)
Porosity       = nc_data_PORO.variables['Porosity'][:].data

m, n = Porosity.shape; z = doy2-doy1+1

path_list = ['am', 'pm']
RFI_th = 0.9

#combined the am and pm data sets
ASCAT_SM_am_pm = np.zeros((m, n, z*2), dtype='float64')
SMOS_SM_am_pm  = np.zeros((m, n, z*2), dtype='float64')
SMAP_SM_am_pm  = np.zeros((m, n, z*2), dtype='float64')
SMAP_tb4_am_pm  = np.zeros((m, n, z*2), dtype='float64')

for path in path_list:
    nc_file_SMOS   = nc_save_dir+'/SMOS_IC_2022_'+path+'_R_'+mission_product+'.nc'
    nc_file_ASCAT  = nc_save_dir+'/ASCAT_TUW_2022_'+path+'_R_'+mission_product+'.nc'
    nc_file_SMAP  = nc_save_dir+'/'+mission_product+'_2022_'+path+'.nc'

    # Load only the relevant slice of the data
    nc_data_SMOS = netCDF4.Dataset(nc_file_SMOS)
    sub_SMOS_SM = nc_data_SMOS.variables['SMOS_SM'][:, :, doy1:doy2+1].data
    sub_SMOS_RFI = nc_data_SMOS.variables['SMOS_RFI'][:, :, doy1:doy2+1].data
    sub_SMOS_SM[sub_SMOS_RFI>RFI_th] = np.nan

    nc_data_ASCAT = netCDF4.Dataset(nc_file_ASCAT)
    sub_ASCAT_SWI = nc_data_ASCAT.variables['ASCAT_SM'][:, :, doy1:doy2+1].data
    sub_ASCAT_SM  = sub_ASCAT_SWI/10000*np.expand_dims(Porosity, axis=2)

    nc_data_SMAP = netCDF4.Dataset(nc_file_SMAP)
    if path == 'am':
        sub_SMAP_SM = nc_data_SMAP.variables['soil_moisture'][:, :, doy1:doy2+1].data
        sub_SMAP_tb4 = nc_data_SMAP.variables['tb_qual_flag_4'][:, :, doy1:doy2+1].data
    else:
        sub_SMAP_SM = nc_data_SMAP.variables['soil_moisture_pm'][:, :, doy1:doy2+1].data
        sub_SMAP_tb4 = nc_data_SMAP.variables['tb_qual_flag_4_pm'][:, :, doy1:doy2+1].data
        
    if path == 'am':
        SMOS_SM_am_pm[:, :,  ::2]  = sub_SMOS_SM
        ASCAT_SM_am_pm[:, :, ::2]  = sub_ASCAT_SM
        SMAP_SM_am_pm[:, :, ::2]   = sub_SMAP_SM
        SMAP_tb4_am_pm[:, :, ::2]  = sub_SMAP_tb4
    else:
        SMOS_SM_am_pm[:, :, 1::2]  = sub_SMOS_SM
        ASCAT_SM_am_pm[:, :, 1::2] = sub_ASCAT_SM
        SMAP_SM_am_pm[:, :, 1::2]  = sub_SMAP_SM
        SMAP_tb4_am_pm[:, :, ::2]  = sub_SMAP_tb4

## 4. Some analysis

In [ ]:
ma_SMOS_SM_am_pm  = Data.moving_average_3d(SMOS_SM_am_pm, 5)
ma_ASCAT_SM_am_pm = Data.moving_average_3d(ASCAT_SM_am_pm, 5)
ma_SMAP_SM_am_pm = Data.moving_average_3d(SMAP_SM_am_pm, 5)

In [ ]:
Plot.plot_global_map(domain_lon, domain_lat, ma_SMOS_SM_am_pm[:,:,1], r'SMOS VSM ($\frac{m^3}{m^3}$)', 0, 0.5, 'jet_r')
Plot.plot_global_map(domain_lon, domain_lat, ma_ASCAT_SM_am_pm[:,:,1], r'ASCAT VSM ($\frac{m^3}{m^3}$)', 0, 0.5, 'jet_r')
Plot.plot_global_map(domain_lon, domain_lat, ma_SMAP_SM_am_pm[:,:,1], r'SMAP VSM ($\frac{m^3}{m^3}$)', 0, 0.5, 'jet_r')

# Display the map with plot_map
bound = [125.7, 129.7, 33.9, 38.8] # entire Korea
#bound = [126.73, 126.95, 35.17, 35.37] # Gwangju
target = ma_SMOS_SM_am_pm[:,:,1]
Plot.plot_map(domain_lon, domain_lat, target, r'SMOS VSM ($\frac{m^3}{m^3}$)', 0, 0.5, 'jet_r', bound)
target = ma_ASCAT_SM_am_pm[:,:,1]
Plot.plot_map(domain_lon, domain_lat, target, r'ASCAT VSM ($\frac{m^3}{m^3}$)', 0, 0.5, 'jet_r', bound)
target = ma_SMAP_SM_am_pm[:,:,1]
Plot.plot_map(domain_lon, domain_lat, target, r'SMAP VSM ($\frac{m^3}{m^3}$)', 0, 0.5, 'jet_r', bound)

## 4.1 Time-series analysis

In [ ]:
seoul_coords = (126.9780, 37.5665)  # Longitude, latitude
gist_coords = (126.8394, 35.2276)

# Plotting the time series for Seoul
Plot.plot_time_series(gist_coords, domain_lon, domain_lat, ma_SMOS_SM_am_pm, 'ma_SMOS')
Plot.plot_time_series(gist_coords, domain_lon, domain_lat, SMOS_SM_am_pm, 'or_SMOS')

Plot.plot_time_series(gist_coords, domain_lon, domain_lat, ma_ASCAT_SM_am_pm, 'ma_ASCAT')
Plot.plot_time_series(gist_coords, domain_lon, domain_lat, ASCAT_SM_am_pm, 'or_ASCAT')

Plot.plot_time_series(gist_coords, domain_lon, domain_lat, ma_SMAP_SM_am_pm, 'ma_SMAP')
Plot.plot_time_series(gist_coords, domain_lon, domain_lat, SMAP_SM_am_pm, 'or_SMAP')

# Finalizing the plot
plt.legend()
plt.title('Soil Moisture MA vs Original Time Series')
plt.grid(True) 
plt.show()

In [ ]:
def extract_region_from_data(longitude, latitude, X, bounds):
    """
    Create a subset of a 3D array based on given latitude and longitude bounds.
    
    Args:
    - X: The 3D array to subset. The first two dimensions should correspond to latitude and longitude.
    - latitude: 2D array of latitude values corresponding to the first dimension of X.
    - longitude: 2D array of longitude values corresponding to the second dimension of X.
    - bounds: Tuple of (lon_min, lon_max, lat_min, lat_max).
    
    Returns:
    - A subset of X corresponding to the specified bounds.
    """
    lon_min, lon_max, lat_min, lat_max = bounds
    
    # Find indices for the bounding box
    lat_indices = np.where((latitude >= lat_min) & (latitude <= lat_max))
    lon_indices = np.where((longitude >= lon_min) & (longitude <= lon_max))

    # Find the minimum and maximum indices to slice the array
    lat_min_idx, lat_max_idx = min(lat_indices[0]), max(lat_indices[0])
    lon_min_idx, lon_max_idx = min(lon_indices[1]), max(lon_indices[1])

    # Subset the array
    subset = X[lat_min_idx:lat_max_idx+1, lon_min_idx:lon_max_idx+1, :]
    
    return subset

def average_over_space(X):
    """
    Compute the average over the first two dimensions in a 3D array.
    
    Args:
    - X: The 3D array to compute the average on. The first two dimensions are averaged.
    
    Returns:
    - A 1D array of shape (Z,) representing the average over the first two dimensions for each layer in the third dimension.
    """
    # Compute the mean over the first two dimensions (latitude and longitude)
    mean_values = np.nanmean(X, axis=(0, 1))
    
    return mean_values

In [ ]:
ROK_bound = (126, 129, 33, 38)
ex_SMAP_SM_am_pm = Data.extract_region_from_data(domain_lon, domain_lat, SMAP_SM_am_pm, ROK_bound)
ex_ma_SMAP_SM_am_pm = Data.extract_region_from_data(domain_lon, domain_lat, ma_SMAP_SM_am_pm, ROK_bound)
ROK_SMAP_SM_am_pm = Data.average_over_space(ex_SMAP_SM_am_pm)
ROK_ma_SMAP_SM_am_pm = Data.average_over_space(ex_ma_SMAP_SM_am_pm)

Plot.plot_time_series(gist_coords, domain_lon, domain_lat, ROK_ma_SMAP_SM_am_pm, 'ROK_ma_SMAP_SM')
Plot.plot_time_series(gist_coords, domain_lon, domain_lat, ROK_SMAP_SM_am_pm, 'ROK_or_SMAP_SM')
# Finalizing the plot
plt.legend()
plt.title('Soil Moisture MA vs Original Time Series over ROK')
plt.grid(True) 
plt.show()

### 4.1 RFI example (advanced)

In [ ]:
# RFI example
def check_Stokes_flags(value, flag_value):
    return (value & flag_value) == flag_value

# Example usage:
value = 65534  # This is the value you got from the 'tb_qual_flag_4' dataset
flag_masks = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 4096, 16384, 32768]

flag_meanings = [
    "Stokes_quality",
    "Stokes_range",
    "Stokes_RFI_detection",
    "Stokes_RFI_correction",
    "Stokes_NEDT",
    "Stokes_direct_sun_correction",
    "Stokes_reflected_sun_correction",
    "Stokes_reflected_moon_correction",
    "Stokes_direct_galaxy_correction",
    "Stokes_reflected_galaxy_correction",
    "Stokes_atmosphere_correction",
    "Stokes_null_value_bit",
    "Stokes_RFI_check",
    "Stokes_RFI_clean"
]

for flag, meaning in zip(flag_masks, flag_meanings):
    if check_Stokes_flags(value, flag):
        print(f"{meaning} flag is set")
    else:
        print(f"{meaning} flag is not set")

In [ ]:
# time-series RFI analysis
seoul_coords = (126.9780, 37.5665)
gist_coords = (126.8394, 35.2276)

rfi_flag_mask = 4  # Replace with the actual mask value for RFI detection
nan_placeholder = -1  # An integer placeholder for NaN values

# Find the closest pixel to Seoul
closest_pixel_index = Data.find_closest_index(domain_lon, domain_lat, seoul_coords)
#closest_pixel_index = Data.find_closest_index(domain_lon, domain_lat, gist_coords)

# Extract the time series of SMAP_tb4_flag values for the closest pixel

tb4_flag_timeseries = SMAP_tb4_am_pm[closest_pixel_index[0], closest_pixel_index[1], :]

# Replace NaN values with the placeholder, and ensure data is an integer type
tb4_flag_timeseries_nonan = np.nan_to_num(tb4_flag_timeseries, nan=nan_placeholder).astype(np.int64)

# Initialize rfi_detected array with zeros (no RFI detected by default)
rfi_detected = np.zeros(tb4_flag_timeseries_nonan.shape, dtype=float)

# Check if RFI is detected (bitwise operation), and mark it as 1 in rfi_detected
rfi_detected[(tb4_flag_timeseries_nonan & rfi_flag_mask) == rfi_flag_mask] = 1

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(rfi_detected, label='RFI Detection Status', linestyle='-', marker='o', markersize=4)
plt.xlabel('Time')
plt.ylabel('RFI Detection (1: Detected, 0: Not Detected)')
plt.title(f'RFI Detection over Time for Pixel Closest to Seoul, Korea')
plt.legend()
plt.grid(True)
plt.ylim(-0.1, 1.1)  # Set y-axis limits to make peaks more visible
plt.show()